In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import numpy as np
import importlib
import pandas as pd
import seaborn
from IPython.display import Image
import matplotlib.pyplot as plt

import Transformer as tnsf
import detection_model as ad

importlib.reload(ad)
importlib.reload(tnsf)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [2]:
pd.options.display.width = 400
pd.options.display.max_colwidth = 400

# Load Dataset

In [3]:
hdfs_template = pd.read_csv("C:\\01. Monash University\\controlroom-xai-alarm-support-transformer\\notebooks\\Spell_result\\HDFS_templates.csv")

hdfs_template

,EventId,EventTemplate
0,E1,[*]Adding an already existing block[*]
1,E2,[*]Verification succeeded for[*]
2,E3,[*]Served block[*]to[*]
3,E4,[*]Got exception while serving[*]to[*]
4,E5,[*]Receiving block[*]src:[*]dest:[*]
5,E6,[*]Received block[*]src:[*]dest:[*]of size[*]
6,E7,[*]writeBlock[*]received exception[*]
7,E8,[*]PacketResponder[*]for block[*]Interrupted[*]
8,E9,[*]Received block[*]of size[*]from[*]
9,E10,[*]PacketResponder[*]Exception[*]


In [4]:
anomaly_label = pd.read_csv("C:\\01. Monash University\\controlroom-xai-alarm-support-transformer\\log_anomaly_detection\\XAI_Anomaly_Detection_Transformer\\Dataset\\HDFS\\Loghub\\anomaly_label.csv",
    names=['BlockId', 'Anomaly_Label']
)

anomaly_label.head()

,BlockId,Anomaly_Label
0,BlockId,Label
1,blk_-1608999687919862906,Normal
2,blk_7503483334202473044,Normal
3,blk_-3544583377289625738,Anomaly
4,blk_-9073992586687739851,Normal


In [5]:
event_traces = pd.read_csv("C:\\01. Monash University\\controlroom-xai-alarm-support-transformer\\log_anomaly_detection\\XAI_Anomaly_Detection_Transformer\\Dataset\\HDFS\\Loghub\\Event_traces.csv")

event_traces.head()

,BlockId,Label,Type,Features,TimeInterval,Latency
0,blk_-1608999687919862906,Success,NaN,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26,E6,E5,E16,E6,E5,E18,E25,E26,E26,E3,E25,E6,E6,E5,E5,E16,E18,E26,E26,E5,E6,E5,E16,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E25,E6,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E25,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3802
1,blk_7503483334202473044,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E3,E2,E2,E23,E23,E23,E21,E21,E21]","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 278.0, 442.0, 3044.0, 0.0, 0.0, 31.0, 0.0, 2.0]",3802
2,blk_-3544583377289625738,Fail,21.0,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E...","[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3797
3,blk_-9073992586687739851,Success,NaN,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E2,E2,E2,E23,E23,E23,E21,E21,E21]","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 28.0, 188.0, 186.0, 49727.0, 0.0, 0.0, 150.0, 24.0, 144.0]",50448
4,blk_7854771516489510256,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E2,E2,E2,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E3,E4,E4,E4,E23,E23,E23,E21,E21,E21]","[0.0, 0.0, 1.0, 48.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 94.0, 285.0, 12.0, 3481.0, 136.0, 2.0, 0.0, 1.0, 1.0, 6.0, 0.0, 0.0, 2.0, 5.0, 1.0, 1.0, 4.0, 4.0, 7.0, 46034.0, 0.0, 0.0, 264.0, 77.0, 117.0]",50583


## Merge Datasets

In [6]:
anomaly_event_traces = event_traces.merge(anomaly_label, on='BlockId', how='left')

anomaly_event_traces.head()

,BlockId,Label,Type,Features,TimeInterval,Latency,Anomaly_Label
0,blk_-1608999687919862906,Success,NaN,"[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26,E6,E5,E16,E6,E5,E18,E25,E26,E26,E3,E25,E6,E6,E5,E5,E16,E18,E26,E26,E5,E6,E5,E16,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E25,E6,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E25,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3802,Normal
1,blk_7503483334202473044,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E3,E2,E2,E23,E23,E23,E21,E21,E21]","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 278.0, 442.0, 3044.0, 0.0, 0.0, 31.0, 0.0, 2.0]",3802,Normal
2,blk_-3544583377289625738,Fail,21.0,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E...","[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3797,Anomaly
3,blk_-9073992586687739851,Success,NaN,"[E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E2,E2,E2,E23,E23,E23,E21,E21,E21]","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 28.0, 188.0, 186.0, 49727.0, 0.0, 0.0, 150.0, 24.0, 144.0]",50448,Normal
4,blk_7854771516489510256,Success,NaN,"[E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26,E2,E2,E2,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E4,E3,E4,E4,E4,E23,E23,E23,E21,E21,E21]","[0.0, 0.0, 1.0, 48.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 94.0, 285.0, 12.0, 3481.0, 136.0, 2.0, 0.0, 1.0, 1.0, 6.0, 0.0, 0.0, 2.0, 5.0, 1.0, 1.0, 4.0, 4.0, 7.0, 46034.0, 0.0, 0.0, 264.0, 77.0, 117.0]",50583,Normal


# Time-Related

In [7]:
df = anomaly_event_traces.copy()

def parse_events(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    if x.startswith("[") and x.endswith("]"):
        x = x[1:-1]
    return [e.strip() for e in x.split(",") if e.strip()]

df["event_list"] = df["Features"].apply(parse_events)

# Start & End
df["Start_EventId"] = df["event_list"].apply(lambda x: x[0] if len(x) > 0 else None)
df["Ending_EventId"] = df["event_list"].apply(lambda x: x[-1] if len(x) > 0 else None)

df[["BlockId", "Start_EventId", "Ending_EventId"]].head()

,BlockId,Start_EventId,Ending_EventId
0,blk_-1608999687919862906,E5,E21
1,blk_7503483334202473044,E5,E21
2,blk_-3544583377289625738,E5,E20
3,blk_-9073992586687739851,E5,E21
4,blk_7854771516489510256,E5,E21


In [8]:
# rename for merging
template = hdfs_template.copy()

start_template = template.rename(columns={
    "EventId": "Start_EventId",
    "EventTemplate": "Start_Template"
})

end_template = template.rename(columns={
    "EventId": "Ending_EventId",
    "EventTemplate": "Ending_Template"
})

# merge
df = df.merge(start_template, on="Start_EventId", how="left")
df = df.merge(end_template, on="Ending_EventId", how="left")

df[[
    "Start_EventId", "Start_Template",
    "Ending_EventId", "Ending_Template"
]].head()

,Start_EventId,Start_Template,Ending_EventId,Ending_Template
0,E5,[*]Receiving block[*]src:[*]dest:[*],E21,[*]Deleting block[*]file[*]
1,E5,[*]Receiving block[*]src:[*]dest:[*],E21,[*]Deleting block[*]file[*]
2,E5,[*]Receiving block[*]src:[*]dest:[*],E20,[*]Unexpected error trying to delete block[*]BlockInfo not found in volumeMap[*]
3,E5,[*]Receiving block[*]src:[*]dest:[*],E21,[*]Deleting block[*]file[*]
4,E5,[*]Receiving block[*]src:[*]dest:[*],E21,[*]Deleting block[*]file[*]


## Starting Events

In [9]:
(pd.crosstab(df["Start_EventId"], df["Anomaly_Label"], normalize="columns") * 100).sort_values("Anomaly", ascending=False).head(10)

Anomaly_Label,Anomaly,Normal
Start_EventId,,
E5,64.995843,74.724080
E22,34.909134,25.265351
E13,0.059389,0.000000
E26,0.023756,0.010032
E25,0.005939,0.000537
E6,0.005939,0.000000


Both in Anomaly and Normal sequences is mostly start by E5 or E22.

In [10]:
hdfs_template[hdfs_template["EventId"].isin(["E5", "E22"])]

,EventId,EventTemplate
4,E5,[*]Receiving block[*]src:[*]dest:[*]
21,E22,[*]BLOCK* NameSystem[*]allocateBlock:[*]


In [ ]:
topk_hit.groupby("dataset")["target_in_pred_pos"].describe()

,count,mean,std,min,25%,50%,75%,max
dataset,,,,,,,,
abnormal_test,88894.0,2.136781,1.894836,1.0,1.0,1.0,3.0,9.0
normal_test,4523680.0,1.547192,1.357921,1.0,1.0,1.0,1.0,9.0


Semantics Meaning of each EventIDs:  
*  E5 : The start of receiving Data and start writing on disk.
*  E22 : NameSystem is Initialization process of creating new blocks (process) then allocate to some places.

## Ending Events

In [11]:
(pd.crosstab(df["Ending_EventId"], df["Anomaly_Label"], normalize="columns") * 100).sort_values("Anomaly", ascending=False).head(10)

Anomaly_Label,Anomaly,Normal
Ending_EventId,,
E21,51.092766,81.686889
E7,19.289702,0.000000
E20,10.606960,0.016481
E22,9.763630,0.005732
E5,7.762205,0.000000
E26,0.884903,16.933555
E27,0.415726,0.000000
E2,0.077206,1.017694
E29,0.041573,0.000000


There are several findings:  
*  Both in Anomaly and Normal sequences, E21 is top than end the sequences.
*  In Anomaly, followed by E7 then E20.
*  On other hand, In Normal, followed by E26 and others divides evenly below 1%.

In [12]:
hdfs_template[hdfs_template["EventId"].isin(["E21", "E26", "E7", "E20"])]

,EventId,EventTemplate
6,E7,[*]writeBlock[*]received exception[*]
19,E20,[*]Unexpected error trying to delete block[*]BlockInfo not found in volumeMap[*]
20,E21,[*]Deleting block[*]file[*]
25,E26,[*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*]


Semantics Meaning of each EventIDs:    
    
**BOTH**   
*  E21 : Removing block after process (Normal: end of process | Anomaly: Recovery deletion)     
**NORMAL**    
*  E26 : Success process indication (stored or registered)     
**ANOMALY**    
*  E7 : "writeBlock" indicate something went wrong and in the middle of writing process (e.g. network failure, interuption, corruption of data, disk error)
*  E20 : When try doing delete process, the file is not exist (e.g. metadata incosistency between metadata lookup table and actual storage)

## Combination of Start and Ending EventIDs

In [13]:
transition_table = pd.crosstab(
    [df["Start_EventId"], df["Ending_EventId"]],
    df["Anomaly_Label"],
    normalize="columns"
) * 100

transition_table.sort_values(by="Anomaly", ascending=False).head(15)

Anomaly_Label                   Anomaly     Normal
Start_EventId Ending_EventId                      
E5            E21             34.701271  57.920401
E22           E21             16.308350  23.756277
E5            E7              11.188977   0.000000
              E22              9.757691   0.005732
E22           E7               8.100725   0.000000
E5            E20              7.987885   0.000358
E22           E5               7.762205   0.000000
              E20              2.619076   0.016123
E5            E26              0.831453  15.563314
              E27              0.368215   0.000000
              E2               0.059389   0.933856
E13           E21              0.053451   0.000000
E22           E26              0.053451   1.370241
              E27              0.047512   0.000000
E5            E29              0.041573   0.000000

In [14]:
hdfs_template[hdfs_template["EventId"].isin(["E22", "E5", "E21", "E26", "E7", "E20"])]

,EventId,EventTemplate
4,E5,[*]Receiving block[*]src:[*]dest:[*]
6,E7,[*]writeBlock[*]received exception[*]
19,E20,[*]Unexpected error trying to delete block[*]BlockInfo not found in volumeMap[*]
20,E21,[*]Deleting block[*]file[*]
21,E22,[*]BLOCK* NameSystem[*]allocateBlock:[*]
25,E26,[*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*]


*  Mostly Start with E5 and end with E21 (both in Normal and Anomaly sequences) : The usual sequences of event start writing (E5) and end with removing the block (E21)
*  Similar to former one, E22 start and ending with E21
*  In Normal, the next frequent sequences is start E5 and end E26 : healthy sequence start writing and end stored
*  In Anomaly, Top 3 and 4 is start with E5 but end with E7 (block appear) or E22 (it supposed to be a initialization steps but appear in the middle of writing of process not done make it abnormal sequences)

## Latency

***Latency Calculated***

In [15]:
import ast

In [16]:
def parse_events(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    if x.startswith("[") and x.endswith("]"):
        x = x[1:-1]
    return [e.strip() for e in x.split(",") if e.strip()]

def parse_intervals(x):
    if pd.isna(x):
        return []
    return list(ast.literal_eval(str(x)))

df["event_list"] = df["Features"].apply(parse_events)
df["interval_list"] = df["TimeInterval"].apply(parse_intervals)

In [17]:
df["latency_calculated"] = df["interval_list"].apply(sum)

In [18]:
df["latency_diff"] = df["Latency"] - df["latency_calculated"]

# quick check
df[["Latency", "latency_calculated", "latency_diff"]].head()

,Latency,latency_calculated,latency_diff
0,3802,3802.0,0.0
1,3802,3802.0,0.0
2,3797,3797.0,0.0
3,50448,50448.0,0.0
4,50583,50583.0,0.0


In [19]:
print("Max diff:", df["latency_diff"].abs().max())
print("Mean diff:", df["latency_diff"].mean())

Max diff: 0.0
Mean diff: 0.0


It was confirmed that Latency is sum of TimeIntervals between EventIDs

***Statistical of Latency***

In [20]:
df.groupby("Anomaly_Label")["Latency"].agg(["mean", "median"])

,mean,median
Anomaly_Label,,
Anomaly,14089.464960,4920.5
Normal,16870.912341,7303.0


Normal sequences tend to have higher latency because they complete the full HDFS block lifecycle, including replication, verification, and cleanup stages.  
In contrast, many anomalous sequences terminate earlier and therefore accumulate less total latency, although some anomalies may still exhibit large delays due to retries, failures, or timeouts.

***TimeIntervals Pattern***

In [21]:
def max_interval_info(row):
    intervals = row["interval_list"]
    events = row["event_list"]
    
    if not intervals:
        return pd.Series([None, None, None])
    
    max_val = max(intervals)
    idx = intervals.index(max_val)
    
    event_before = events[idx] if idx < len(events) else None
    event_after = events[idx+1] if idx+1 < len(events) else None
    
    return pd.Series([max_val, idx, f"{event_before} -> {event_after}"])

df[["max_interval", "max_interval_idx", "critical_transition"]] = df.apply(
    max_interval_info, axis=1
)

In [22]:
df[[
    "BlockId",
    "Anomaly_Label",
    "max_interval",
    "max_interval_idx",
    "critical_transition"
]].head()

,BlockId,Anomaly_Label,max_interval,max_interval_idx,critical_transition
0,blk_-1608999687919862906,Normal,3703.0,248,E3 -> E23
1,blk_7503483334202473044,Normal,3044.0,15,E2 -> E23
2,blk_-3544583377289625738,Anomaly,3713.0,215,E3 -> E23
3,blk_-9073992586687739851,Normal,49727.0,15,E2 -> E23
4,blk_7854771516489510256,Normal,46034.0,31,E4 -> E23


In [23]:
df.groupby("Anomaly_Label")["max_interval"].agg(["mean", "median", "max"])

,mean,median,max
Anomaly_Label,,,
Anomaly,9198.354377,3483.5,46341.0
Normal,12080.852838,6946.0,53611.0


Similar to the statistical of Latency result where the Normal sequences makes it much longer to process instead of interrupted.

In [24]:
df[df['Anomaly_Label'] == 'Normal']['critical_transition'].value_counts().head()

critical_transition
E26 -> E23    265237
E5 -> E11      52942
E22 -> E11     37633
E9 -> E23      30443
E2 -> E23      28836
Name: count, dtype: int64

In [25]:
hdfs_template[hdfs_template["EventId"].isin(["E5", "E11", "E26", "E23", "E22", "E9", "E2", "E7"])]

,EventId,EventTemplate
1,E2,[*]Verification succeeded for[*]
4,E5,[*]Receiving block[*]src:[*]dest:[*]
6,E7,[*]writeBlock[*]received exception[*]
8,E9,[*]Received block[*]of size[*]from[*]
10,E11,[*]PacketResponder[*]for block[*]terminating[*]
21,E22,[*]BLOCK* NameSystem[*]allocateBlock:[*]
22,E23,[*]BLOCK* NameSystem[*]delete:[*]is added to invalidSet of[*]
25,E26,[*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*]


*  The top frequent E26 -> E23 indicates the writing is success stored, then delete the process block
*  Same with top2 E5 -> E11 indicates receive block writing then terminate after finish
*  Top3 E22 -> E11 indicates items is stored/allocated then terminate after finish
*  Top 4 E9 -> E23 receiving items/files then delete the process block after finish
*  Top 5 E2 -> E23 verification of files/items then delete process block after finish  
   
The similar pattern shown that in Normal Sequences the maximum Interval times reached when the one process is about to finish then delete/terminate is conduct.

In [26]:
df[df['Anomaly_Label'] == 'Anomaly']['critical_transition'].value_counts().head()

critical_transition
E26 -> E23    3786
E5 -> E7      1851
E5 -> E22     1644
E22 -> E5     1393
E22 -> E7     1258
Name: count, dtype: int64

*  Similar occurances appear in some Anomaly sequences like E26 -> E23, where the process is about to finish then continue to being terminate as it was done
*  E5 -> E7 ; E22 -> E7: while writing process conduct, it crash in the middle (exception)
*  E5 -> E22 ; E22 -> E5 ; : writing -> allocate files ; the first process not yet done, another process is triggered

While the most frequent one that get Maximum `TimeIntervals` are E26 -> E23 which indicate usual pattern of process then terminate while finish.  
Other frequents pattern shown that crashed (e.g. E5 -> E7 and E22 -> E7) and interuppted process (E5 -> E22 and E22 -> E5) are the frequent one in Anomaly sequences.